# PredictLeads Discovery and Sample Analysis

This notebook brings the reviewed snapshot findings together with a local analysis of the completed 30,000-job reservoir. It reads documentation and one existing JSONL sample from disk only. **Run All does not access S3, rerun scans, or call AI services.**

## What Is Completed

- Full-file quality audit: 13,415,388 job records and 340,669 company records; both streams reached EOF.
- Full-file job-title/occupation frequency exploration and a seeded uniform 30,000-job sample.
- Reversible preparation helpers and two 10,000-record sequential development slices.

These counts describe records, not verified unique listings or currently open jobs. The source snapshot is historical. Null status is unknown, not open; observation timestamps are not posting or closing dates. The 30,000 rows support sample-level exploration, not whole-dataset prevalence claims.

## Proposed, Not Decided

Career-transition recommendations, skill extraction, cross-file company enrichment, and ML/LLM matching are not implemented or approved team decisions. Preparation fields are derived views alongside source values. An absent skill mention does not prove the skill is unnecessary, and source seniority labels do not establish a candidate's eligibility.

Read `docs/discovery-summary.md`, `docs/artifact-index.md`, and `docs/data-preparation.md` for reviewed findings, artifact paths, and field meanings.

In [ ]:
from pathlib import Path

workspace = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "docs" / "discovery-summary.md").is_file()
     and (candidate / "docs" / "artifact-index.md").is_file()),
    None,
)
if workspace is None:
    print("Project documentation was not found from this notebook's working directory.")
    print("Open this notebook from the project workspace and see README.md for setup.")
else:
    for relative_path in ("docs/discovery-summary.md", "docs/artifact-index.md"):
        print(f"\n{'=' * 16} {relative_path} {'=' * 16}\n")
        print((workspace / relative_path).read_text(encoding="utf-8"))

## Full-Dataset Findings: Completed Runs

The following figures come from the **full-file audit and full-file role exploration**, not from the 30,000-job sample shown later. They are counts of records in the September 30, 2026 historical snapshot, not verified unique jobs or current openings. Null status means availability is unconfirmed.

### Audit Evidence

- The full audit reached EOF with no run failures: **13,415,388 job records** and **340,669 company records**.
- Among jobs, descriptions were null for 2,386,488 records, salary for 7,311,883, location for 1,126,469, status for 4,754,676, and `posted_at` for 7,539,490.
- All 13,415,388 job-company references matched a company resource in the same wrapper using `(type, id)`. This does not establish coverage against the separate companies file.
- Company descriptions were null for 32,820 records and blank for 76; company names were null for 2,170.

### Role-Frequency Evidence

The full jobs stream reached EOF: exact counts cover 13,415,388 eligible job records. Most frequent raw titles included Customer Service Representative (25,231), Sales Associate (23,584), and Maintenance Technician (23,300). Most frequent O*NET occupations included Registered Nurses (346,100), Retail Salespersons (229,808), and Software Developers (222,674). These are exact **record frequencies**, not unique-role estimates.

Detailed evidence: [discovery summary](../docs/discovery-summary.md), [full audit report](../data/discovery/full_audit_run_20261005_134741/full_audit/full_quality_report.md), [role exploration report](../data/discovery/role_exploration/roles_corrected_20261006T000000Z/role_exploration_report.md), and [artifact index](../docs/artifact-index.md).

### Explore Saved Full-Run Frequency Exports

This cell reads only existing aggregate CSV exports, if present. The title/occupation/family/pair counts below are **full-population export rows**, selected by the CSV `scope`; they are not recalculated from the sample. If the local exports are missing, use the report links above and ask the team for the approved local artifact copy. No scan is launched.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

ROLE_EXPORT_DIR = Path("data/discovery/role_exploration/roles_corrected_20261006T000000Z")
if not ROLE_EXPORT_DIR.is_absolute():
    ROLE_EXPORT_DIR = workspace / ROLE_EXPORT_DIR if workspace else Path.cwd() / ROLE_EXPORT_DIR

full_export_specs = {
    "raw_titles.csv": ("Full-population raw titles", 12),
    "occupations.csv": ("Full-population O*NET occupations", 12),
    "families.csv": ("Full-population O*NET families", 12),
    "normalized_title_occupation_pairs.csv": ("Full-population normalized-title / occupation pairs", 12),
}
full_export_tables = {}
for filename, (label, limit) in full_export_specs.items():
    export_path = ROLE_EXPORT_DIR / filename
    if not export_path.is_file():
        print(f"Saved aggregate export unavailable: {export_path}")
        print("Use the full role exploration report above, or obtain the approved local artifact copy.")
        continue
    # Exports are ordered by scope then descending frequency. Read a bounded prefix,
    # then retain population rows explicitly; never substitute sample-scope rows.
    exported = pd.read_csv(export_path, encoding="utf-8-sig", nrows=limit + 20)
    population_rows = exported.loc[exported["scope"].eq("population")].head(limit).copy()
    if population_rows.empty:
        print(f"No population-scope rows found in {filename}; not presenting sample rows as full counts.")
        continue
    full_export_tables[filename] = population_rows
    print(label, f"(saved exact full-run record counts; {len(population_rows)} rows shown):")
    display(population_rows)

## Possible ML Experiments: Evidence, Hypotheses, Limits

**Evidence:** The full snapshot contains rich title/description/category/location fields and exact title-to-O*NET frequency associations. Description is null for 2.39 million jobs, salary for 7.31 million, and status for 4.75 million. The role counts describe records; the source `seniority` field is a title-derived label.

**Hypotheses to test, not conclusions:** A text-retrieval baseline could rank listings against human-authored queries; a title/occupation graph could suggest adjacent role labels for investigation; and an evidence-grounded skill extraction prototype could compare extracted mentions with human annotations. These experiments do not choose a target audience or imply that a model will perform well.

**Evaluation and limits:** Define representative, consented or synthetic queries and human relevance labels before fitting or tuning; compare simple baselines using precision@k/recall@k or nDCG, then test subgroup and missing-data behavior. This workspace has no relevance/outcome labels, candidate profiles, approved skill taxonomy, separate-file company join, or model evaluation. A skill absent from text is not proven unnecessary; missing descriptions/status, historical freshness, duplicate/version semantics, possible source bias, and non-unique record frequencies constrain any conclusion. Keep proposals subject to team review.

## Load the Existing Local Sample

The default is the completed seeded sample at `data/discovery/role_exploration/roles_corrected_20261006T000000Z/jobs_prepared_random_sample.jsonl`. Put your approved local copy at that path, or edit `SAMPLE_PATH` in the next cell. Keep the file under ignored `data/`; do not add it to Git. Install notebook dependencies with `python -m pip install -r requirements.txt` in the project environment.

The loader reads prepared JSONL records with pandas. If the file is missing, it explains where to obtain/place the local copy and creates an empty frame so the remaining cells can complete harmlessly during Run All.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

SAMPLE_PATH = Path("data/discovery/role_exploration/roles_corrected_20261006T000000Z/jobs_prepared_random_sample.jsonl")
if not SAMPLE_PATH.is_absolute():
    SAMPLE_PATH = workspace / SAMPLE_PATH if "workspace" in globals() and workspace else Path.cwd() / SAMPLE_PATH


def load_local_sample(path, *, announce=True):
    if path.is_file():
        return pd.read_json(path, lines=True, dtype=False), True
    if announce:
        print(f"Local sample not found: {path}")
        print("Obtain the approved sample copy from your team, place it at this path, or edit SAMPLE_PATH above.")
        print("The notebook does not download source data or regenerate the sample.")
    return pd.DataFrame(), False


jobs, sample_available = load_local_sample(SAMPLE_PATH)
if sample_available:
    print(f"Loaded {len(jobs):,} prepared job records from {SAMPLE_PATH}")

_missing_test_path = SAMPLE_PATH.with_name(SAMPLE_PATH.name + ".not_present_test")
_missing_test_jobs, _missing_test_available = load_local_sample(_missing_test_path, announce=False)
assert not _missing_test_available and _missing_test_jobs.empty
print("Missing-file fallback check passed (no data access attempted).")

## Sample Overview and Field Guide

All statistics and charts below are calculated from the loaded 30,000-record reservoir only. They do not replace the full-file audit or exact full-dataset frequency tables. Preview rows contain a short set of fields; descriptions and identifiers are omitted from the preview.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

if not sample_available:
    print("Sample analysis is skipped until the local JSONL copy is available.")
else:
    print(f"Sample records: {len(jobs):,}")
    print(f"Available columns ({len(jobs.columns)}):")
    print(", ".join(map(str, jobs.columns)))

    missing_table = (
        jobs.isna().sum().rename("missing_records").to_frame()
        .assign(missing_percent=lambda table: (100 * table["missing_records"] / len(jobs)).round(2))
        .sort_values(["missing_records"], ascending=False)
    )
    print("Missing-value counts (pandas nulls; empty strings/arrays are not null):")
    display(missing_table)

    field_guide = pd.DataFrame([
        ("title_raw", "Source title as supplied."),
        ("title_normalized", "Source-provided normalized title."),
        ("title_translated", "Source-provided translated title."),
        ("title_searchable", "Derived fallback: normalized, then translated, then raw title."),
        ("onet_data.occupation_name", "Source O*NET occupation association."),
        ("seniority", "Source-provided seniority inference; not candidate eligibility."),
        ("location_raw", "Source location text; structured details are separately retained."),
        ("status_raw", "Source status value; null is not proof of an open listing."),
        ("status_derived", "Derived closed only for explicit closed; otherwise unknown."),
        ("description_raw", "Original source description; not rendered in preview tables."),
        ("description_text", "Derived readable HTML/Markdown text projection."),
        ("quality_flags", "Derived field-state, relationship, and date-quality indicators."),
        ("company_*_included", "Company details from same-wrapper included resource, if matched."),
    ], columns=["field", "meaning"])
    print("Field meanings:")
    display(field_guide)

    def _sample_text(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return "<missing>"
        if isinstance(value, (dict, list)):
            return json.dumps(value, ensure_ascii=False, sort_keys=True)
        text = str(value).strip()
        return text if text else "<blank>"

    sample_view = pd.DataFrame(index=jobs.index)
    sample_view["source_title"] = jobs.get("title_raw", pd.Series(index=jobs.index, dtype="object")).map(_sample_text)
    sample_view["occupation"] = jobs.get("onet_data", pd.Series(index=jobs.index, dtype="object")).map(
        lambda value: _sample_text(value.get("occupation_name") if isinstance(value, dict) else None)
    )
    for source_column, view_column in (("seniority", "seniority"), ("location_raw", "location"),
                                       ("status_derived", "status")):
        sample_view[view_column] = jobs.get(source_column, pd.Series(index=jobs.index, dtype="object")).map(_sample_text)
    sample_view["description_availability"] = jobs.get(
        "description_usable", pd.Series(False, index=jobs.index)
    ).map(lambda value: "available" if bool(value) else "unavailable")
    print("Prepared-record preview (first 8 rows; values are from the sample):")
    display(sample_view.head(8).reset_index(drop=True))

    def _top_counts(series, limit=12):
        counts = series.map(_sample_text).value_counts(dropna=False)
        top = counts.head(limit)
        for state in ("<missing>", "<blank>"):
            if state in counts.index and state not in top.index:
                top = pd.concat([top, counts.loc[[state]]])
        return top

    chart_specs = [
        ("Common searchable titles", jobs.get("title_searchable", pd.Series(index=jobs.index, dtype="object"))),
        ("Common O*NET occupations", jobs.get("onet_data", pd.Series(index=jobs.index, dtype="object")).map(
            lambda value: value.get("occupation_name") if isinstance(value, dict) else None
        )),
        ("Seniority", jobs.get("seniority", pd.Series(index=jobs.index, dtype="object"))),
        ("Location", jobs.get("location_raw", pd.Series(index=jobs.index, dtype="object"))),
        ("Status (derived)", jobs.get("status_derived", pd.Series(index=jobs.index, dtype="object"))),
        ("Description availability", jobs.get("description_usable", pd.Series(False, index=jobs.index)).map(
            lambda value: "available" if bool(value) else "unavailable"
        )),
    ]
    distribution_rows = []
    for dimension, values in chart_specs:
        for value, count in _top_counts(values).items():
            distribution_rows.append({"dimension": dimension, "value": value, "sample_records": count})
    print("Most common sample values (top values per dimension; `<missing>` and `<blank>` are retained):")
    display(pd.DataFrame(distribution_rows))

    figure, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
    for axis, (title, values) in zip(axes.flat, chart_specs):
        counts = _top_counts(values)
        counts.sort_values().plot(kind="barh", ax=axis, color="#287271")
        axis.set_title(title, fontsize=11)
        axis.set_xlabel("Sample records", fontsize=9)
        axis.set_ylabel("")
        axis.set_yticklabels([label.get_text()[:34] for label in axis.get_yticklabels()], fontsize=8)
        axis.tick_params(axis="x", labelsize=8)
        axis.grid(axis="x", alpha=0.2)
    figure.suptitle("30,000-job sample only; missing/blank values remain visible", fontsize=14)
    plt.show()
    plt.close(figure)

## Edit Filters

Set any filter string below. Empty strings mean unrestricted. Matching is literal and case-insensitive, so punctuation such as `C++` is treated as text rather than a regular expression. Multiple terms are not required; each field accepts one substring.

In [ ]:
# Edit these literal substring filters; blanks leave each dimension unrestricted.
TITLE_DESCRIPTION_KEYWORD = ""
OCCUPATION_KEYWORD = ""
SENIORITY_KEYWORD = ""
LOCATION_KEYWORD = ""

if not sample_available:
    filtered_jobs = pd.DataFrame()
    print("Filters are idle because the local sample is unavailable.")
else:
    def _contains_literal(series, keyword):
        if not keyword.strip():
            return pd.Series(True, index=series.index)
        safe = series.map(_sample_text)
        return safe.str.contains(re.escape(keyword.strip()), case=False, regex=True, na=False)

    filter_check = _contains_literal(pd.Series(["C++ engineer", None, "C# developer"]), "C++")
    assert filter_check.tolist() == [True, False, False]

    occupation_values = jobs.get("onet_data", pd.Series(index=jobs.index, dtype="object")).map(
        lambda value: value.get("occupation_name") if isinstance(value, dict) else None
    )
    assert _contains_literal(occupation_values, "registered nurses").any()
    assert _contains_literal(jobs.get("seniority", pd.Series(index=jobs.index, dtype="object")), "JUNIOR").any()
    location_values = (
        jobs.get("location_raw", pd.Series(index=jobs.index, dtype="object")).map(_sample_text)
        + " " + jobs.get("location_data_raw", pd.Series(index=jobs.index, dtype="object")).map(_sample_text)
    )
    assert _contains_literal(location_values, "united states").any()
    assert not _contains_literal(occupation_values, "__no_such_occupation_filter__").any()

    title_description = (
        jobs.get("title_raw", pd.Series("", index=jobs.index)).map(_sample_text)
        + "\n" + jobs.get("description_text", pd.Series("", index=jobs.index)).map(_sample_text)
    )
    mask = (
        _contains_literal(title_description, TITLE_DESCRIPTION_KEYWORD)
        & _contains_literal(occupation_values, OCCUPATION_KEYWORD)
        & _contains_literal(jobs.get("seniority", pd.Series(index=jobs.index, dtype="object")), SENIORITY_KEYWORD)
        & _contains_literal(location_values, LOCATION_KEYWORD)
    )
    filtered_jobs = jobs.loc[mask].copy()
    print("Filter checks passed: literal, case-insensitive, null-safe, and non-empty/no-match terms.")
    print(f"Matching records: {len(filtered_jobs):,} of {len(jobs):,}")
    filtered_preview = pd.DataFrame({
        "source_title": filtered_jobs.get("title_raw", pd.Series(index=filtered_jobs.index, dtype="object")),
        "occupation": filtered_jobs.get("onet_data", pd.Series(index=filtered_jobs.index, dtype="object")).map(
            lambda value: value.get("occupation_name") if isinstance(value, dict) else None
        ),
        "seniority": filtered_jobs.get("seniority"),
        "location": filtered_jobs.get("location_raw"),
        "status": filtered_jobs.get("status_derived"),
        "description_available": filtered_jobs.get("description_usable"),
    })
    display(filtered_preview.head(10).reset_index(drop=True))

## Inspect One Filtered Record

Set `SELECTED_FILTERED_POSITION` to a zero-based row position in the filtered results. The view separates source fields from derived preparation fields, includes same-wrapper company details and quality flags, and caps readable description text at 3,000 characters. Do not share job-level outputs without clearing notebook outputs first.

In [ ]:
SELECTED_FILTERED_POSITION = 0
DESCRIPTION_DISPLAY_LIMIT = 3000

if not sample_available or filtered_jobs.empty:
    print("No filtered record is available. Load the sample and adjust filters first.")
elif not 0 <= SELECTED_FILTERED_POSITION < len(filtered_jobs):
    print(f"Choose a position from 0 to {len(filtered_jobs) - 1:,}.")
else:
    selected = filtered_jobs.iloc[SELECTED_FILTERED_POSITION]

    def _value_or_missing(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return "<missing>"
        if isinstance(value, (dict, list)):
            return json.dumps(value, ensure_ascii=False, indent=2)
        text = str(value)
        return text if text.strip() else "<blank>"

    source_fields = ["title_raw", "title_normalized", "title_translated", "status_raw", "location_raw", "salary_raw"]
    derived_fields = ["title_searchable", "title_searchable_source", "status_derived", "description_usable", "historical_snapshot", "current_opening_verified"]
    company_fields = ["company_match_state", "company_reference", "company_name_included", "company_domain_included", "company_match_count"]

    print("Source values (as retained from the source or same-wrapper included resource):")
    display(pd.DataFrame([{"field": field, "value": _value_or_missing(selected.get(field))} for field in source_fields]))
    print("Derived preparation fields:")
    display(pd.DataFrame([{"field": field, "value": _value_or_missing(selected.get(field))} for field in derived_fields]))
    print("Same-wrapper company details and match state (no separate companies-file join):")
    display(pd.DataFrame([{"field": field, "value": _value_or_missing(selected.get(field))} for field in company_fields]))

    readable_description = selected.get("description_text")
    if readable_description is None or pd.isna(readable_description):
        print("Readable description: <unavailable>")
    else:
        readable_description = str(readable_description)
        print("Readable description (derived from source description):")
        print(readable_description[:DESCRIPTION_DISPLAY_LIMIT])
        if len(readable_description) > DESCRIPTION_DISPLAY_LIMIT:
            print(f"[truncated; {len(readable_description):,} characters total]")

    flags = selected.get("quality_flags")
    if isinstance(flags, dict):
        print("Quality flags (derived):")
        display(pd.DataFrame([{"field": field, "state_or_flag": value} for field, value in flags.items()]))
    else:
        print("Quality flags: <unavailable>")

    print("Interpretation: this is a historical listing. A skill not mentioned in the description is not necessarily unnecessary, and the source seniority label does not establish candidate eligibility.")